In [3]:
from dotenv import load_dotenv, find_dotenv

In [4]:
load_dotenv(find_dotenv())

True

In [1]:
from langchain_core.documents import Document

docs = [
    Document(page_content="پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند. قابل شستشو با ماشین لباسشویی.", metadata={"source": "catalog", "id": 1}),
    Document(page_content="مایوهای شنای ما بسیار راحت هستند و سریع خشک می‌شوند. موجود در ۶ رنگ.", metadata={"source": "catalog", "id": 2}),
    Document(page_content="چکمه‌های کوهنوردی ضدآب هستند و پشتیبانی عالی از مچ پا در مسیرهای پیاده‌روی ارائه می‌دهند.", metadata={"source": "catalog", "id": 3}),
    Document(page_content="کلاه محافظ خورشید اشعه‌های فرابنفش را مسدود می‌کند و لبه پهنی دارد. طراحی سبک.", metadata={"source": "catalog", "id": 4}),
    Document(page_content="ژاکت پشمی در هوای سرد گرما را حفظ می‌کند. قابل شستشو با ماشین لباسشویی و با قابلیت بسته‌بندی آسان.", metadata={"source": "catalog", "id": 5}),
]

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

splits = splitter.split_documents(docs)

In [5]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

sample_embedded = embeddings.embed_query("پیراهن محافظ خورشید")

In [6]:
from langchain_community.vectorstores import FAISS

# ساخت vector store
vector_store = FAISS.from_documents(splits, embeddings)

# تست similarity search
query = "پیراهن هایی با محافظت در برابر نور خورشید"
results = vector_store.similarity_search(query, k=2)
for i, doc in enumerate(results):
    print(f"  {i+1}. {doc.page_content[:100]}")

C:\Users\Taghavi\AppData\Local\Temp\ipykernel_13656\3382575494.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


  1. پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند. قابل 
  2. کلاه محافظ خورشید اشعه‌های فرابنفش را مسدود می‌کند و لبه پهنی دارد. طراحی سبک.


In [7]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# With Similarity Score
results = vector_store.similarity_search_with_relevance_scores(query, k=2)

df = []

for i, (doc, score) in enumerate(results):
    df.append({
        "Document":doc.page_content,
        "Similarity":f"{score:.3f}"
    })

pd.DataFrame(df)

,Document,Similarity
0,پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند. قابل شستشو با ماشین لباسشویی.,0.603
1,کلاه محافظ خورشید اشعه‌های فرابنفش را مسدود می‌کند و لبه پهنی دارد. طراحی سبک.,0.362


In [8]:
# Bad Version

from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 1. Define LLM
llm = init_chat_model(model="gpt-4o-mini",model_provider="openai",temperature=0)

# 2. Create Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system","""
    جواب سوالات کاربر رو فقط و فقط بر اساس context بده

    Context:
    {context}
    """),
    ("human","{question}")
])

# 3. Generate Context
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Chain
chain = prompt | llm | StrOutputParser()

# 5. Ask from LLM
query = "لطفاً تمام پیراهن های دارای محافظت در برابر خورشید را لیست کنید"
retriever_results = format_docs(vector_store.similarity_search(query=query,k=2))

response = chain.invoke({"context":retriever_results,"question":query})
print(response)


پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند.


In [9]:
# Good Version

from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages([
    ("system","""
    جواب سوالات کاربر رو فقط و فقط بر اساس context بده

    Context:
    {context}
    """),
    ("human","{question}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

context = retriever | format_docs

chain = (
   { "context":context,"question":RunnablePassthrough()} | prompt | llm | StrOutputParser()
)

response = chain.invoke("لطفاً تمام پیراهن های دارای محافظت در برابر خورشید را لیست کنید")
print(response)

پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند.


### مثال های تکمیلی برای Runnable

In [ ]:
from langchain_core.runnables import RunnableLambda

# ------------------------- Runabble -------------------------
# 1.
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)
query = "پیراهن ضد آفتاب دارید؟"
docs = retriever.invoke(query)
print(docs[0].page_content)

# 2. چون خروجی سیمیلاریتی سرچ ران ابل نیست که اینوک داشته باشه. با این روش ران ابلش میکنیم که بتونیم با پای مقدارشو پاس بدیم به متد بعدی
search = RunnableLambda(
    lambda query: vector_store.similarity_search(query, k=3)
)
docs = search.invoke("پیراهن ضد آفتاب دارید؟")
print(docs[0].page_content)
# ------------------------- Runabble -------------------------

# -------------------------- Chain ---------------------------

# 1. Chain با Retriever
chain = retriever | format_docs
result = chain.invoke("پیراهن ضد آفتاب دارید؟")
print(result)

# 2. Chain با RunnableLambda
chain = search | format_docs
result = chain.invoke("پیراهن ضد آفتاب دارید؟")
print(result)
# -------------------------- Chain ---------------------------

### اضافه کردن متد سرچ بعنوان تول به رگ و ساخت ایجنت

In [10]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool


@tool
def search_catalog(query:str):
    """در کاتالوگ محصولات جست‌وجو کن تا موارد مرتبط را پیدا کنی.
    وقتی کاربر درباره محصولات، پوشاک یا تجهیزات فضای باز سؤال می‌پرسد، از این ابزار استفاده کن."""
    docs = vector_store.similarity_search_with_relevance_scores(query=query,k=2)

    threshold = 0.3

    docs = [doc.page_content for doc,score  in docs if score >= threshold]
    if not docs:
        return "No relevant products found."
    return "\n".join(docs)


agent = create_agent(
    model=llm,
    tools=[search_catalog],
system_prompt = """
تو دستیار هوشمند فروشگاه پوشاک هستی.

قوانین:
1. برای سوالات مربوط به محصولات حتماً از search_catalog استفاده کن.
2. فقط بر اساس نتایج ابزار پاسخ بده.
3. اگر ابزار محصول مرتبطی پیدا نکرد، بگو:
   "متأسفانه محصول مرتبطی در کاتالوگ پیدا نکردم."
4. هرگز محصول یا مشخصاتی خارج از نتایج ابزار اختراع نکن.
"""
)

response = agent.invoke({
    "messages":[HumanMessage("پیراهن ضد آفتاب داری؟")]
})
print(response["messages"][-1].content)


بله، ما پیراهن محافظ خورشید داریم که دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند. این پیراهن قابل شستشو با ماشین لباسشویی نیز می‌باشد. اگر سوال دیگری دارید یا به اطلاعات بیشتری نیاز دارید، خوشحال می‌شوم کمک کنم!
